In [1]:
# Load packages
import json
import numpy as np
import pandas as pd
import pickle
import requests

from nba_api.stats.endpoints import shotchartdetail
from nba_api.stats.static import players, teams

from nba import NbaScraper, ShotCharts

In [2]:
# Variables
YEARS = ['2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24']

In [3]:
# Load players
with open('./data/players_career.pickle', 'rb') as f:
    players_career = pickle.load(f)

In [4]:
# Number of players
print(f"Number of players: {len(players_career)}")

Number of players: 573


In [5]:
# Create a dataset containing the players with the year
# in YEARS in NBA
players = list()
for pp in players_career:
    temp = pp['career']
    temp = temp[temp['SEASON_ID'].isin(YEARS)]
    if len(temp) > 4:
        players.append({
            'PLAYER_ID': pp['id'],
            'full_name': pp['full_name'],
            'career': temp[['PLAYER_ID', 'SEASON_ID', 'TEAM_ID']]
        })
    else:
        continue

In [6]:
players_season_long = pd.concat([pp['career'] for pp in players])
players_season_long = players_season_long.query('TEAM_ID > 0')

In [7]:
players_season_long

,PLAYER_ID,SEASON_ID,TEAM_ID
0,1630173,2020-21,1610612748
1,1630173,2021-22,1610612761
2,1630173,2022-23,1610612761
3,1630173,2023-24,1610612761
4,1630173,2023-24,1610612752
...,...,...,...
5,1627826,2019-20,1610612746
6,1627826,2020-21,1610612746
7,1627826,2021-22,1610612746
8,1627826,2022-23,1610612746


In [8]:
df = pd.DataFrame()
for idx, row in players_season_long.iterrows():
    shot_data = shotchartdetail.ShotChartDetail(
        team_id=row['TEAM_ID'],
        player_id=row['PLAYER_ID'],
        context_measure_simple='FGA',
        season_nullable=row['SEASON_ID']
    )
    df = pd.concat([df, shot_data.get_data_frames()[0]])

0, 1630173
1, 1630173
2, 1630173
3, 1630173
4, 1630173
1, 1628389
2, 1628389
3, 1628389
4, 1628389
5, 1628389
6, 1628389
0, 1629638
1, 1629638
2, 1629638
3, 1629638
5, 1629638
6, 1629638
8, 1629638
0, 1628960
1, 1628960
2, 1628960
3, 1628960
4, 1628960
5, 1628960
1, 1628386
2, 1628386
3, 1628386
4, 1628386
6, 1628386
7, 1628386
8, 1628386
4, 203937
5, 203937
6, 203937
7, 203937
8, 203937
9, 203937
5, 203507
6, 203507
7, 203507
8, 203507
9, 203507
10, 203507
1, 203648
2, 203648
3, 203648
4, 203648
5, 203648
1, 1628384
2, 1628384
3, 1628384
4, 1628384
5, 1628384
6, 1628384
7, 1628384
1, 1627853
2, 1627853
3, 1627853
4, 1627853
5, 1627853
6, 1627853
8, 1627853
0, 1629028
1, 1629028
2, 1629028
3, 1629028
4, 1629028
5, 1629028
0, 1628963
1, 1628963
2, 1628963
3, 1628963
4, 1628963
6, 1628963
7, 1628963
8, 1628963
0, 1628964
1, 1628964
2, 1628964
3, 1628964
4, 1628964
5, 1628964
7, 1628964
0, 1630625
1, 1630625
2, 1630625
3, 1630625
6, 203084
7, 203084
9, 203084
10, 203084
11, 203084
12, 203

In [9]:
# Save shots
with open('./data/player_shots.pickle', 'wb') as f:
    pickle.dump(df, f, protocol=pickle.HIGHEST_PROTOCOL)